In [10]:
import pandas as pd
def calculate_league_win_percentage_dispersion(in_path: str, out_path: str) -> pd.DataFrame:
    """
    Calculate the standard deviation of win percentages per league.
    
    Win percentage = total_points_in_season / (number_of_games * 3)
    Each (season, team) pair is one data point.
    Returns one standard deviation value per league.
    
    Output columns: league, std_win_percentage, num_data_points
    """
    df = pd.read_csv(in_path)
    df["home_team_points"] = df["hometeamresult"].map({1: 3, 0: 1, -1: 0})
    df["away_team_points"] = df["hometeamresult"].map({-1: 3, 0: 1, 1: 0})
    # Check required columns
    need = {"season", "league", "home_team", "away_team", "home_team_points", "away_team_points"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"Missing columns: {sorted(miss)}")
    
    # Create home and away dataframes
    home = pd.DataFrame({
        "league": df["league"],
        "season": df["season"],
        "team": df["home_team"],
        "points": df["home_team_points"]
    })
    
    away = pd.DataFrame({
        "league": df["league"],
        "season": df["season"],
        "team": df["away_team"],
        "points": df["away_team_points"]
    })
    
    # Combine home and away records
    long = pd.concat([home, away], ignore_index=True)
    
    # Calculate total points and games per (season, team, league)
    season_team_stats = (long
        .groupby(["league", "season", "team"], as_index=False)
        .agg(
            total_points=("points", "sum"),
            num_games=("points", "count")
        )
    )
    
    # Calculate win percentage for each (season, team) pair
    season_team_stats["win_percentage"] = (
        season_team_stats["total_points"] / (season_team_stats["num_games"] * 3)
    )
    
    # Calculate standard deviation per league across all (season, team) pairs
    league_dispersion = (season_team_stats
        .groupby("league", as_index=False)
        .agg(
            std_win_percentage=("win_percentage", lambda x: x.std(ddof=0)),
            num_data_points=("win_percentage", "count")
        )
    )
    
    # Sort by league name
    league_dispersion = league_dispersion.sort_values("league").reset_index(drop=True)
    
    # Save to CSV
    league_dispersion.to_csv(out_path, index=False)
    print(f"Wrote: {out_path} (rows: {len(league_dispersion):,})")
    
    return league_dispersion

In [12]:
# run to generate csv files

# Actual
actual = calculate_league_win_percentage_dispersion(
    in_path="/Users/joeyli/skillvsluck/data/european_soccer_leagues/actual/actual_combined_matches.csv",
    out_path="/Users/joeyli/skillvsluck/output/european_soccer_leagues/points_std/actual.csv",)

# Pure Skill
skill = calculate_league_win_percentage_dispersion(
    in_path="/Users/joeyli/skillvsluck/data/european_soccer_leagues/pure_skill/skill_based_league.csv",
    out_path="/Users/joeyli/skillvsluck/output/european_soccer_leagues/points_std/skilled.csv",)

# # Pure Luck (home bias v1)
# luck = calculate_league_win_percentage_dispersion(
#     in_path="../../data/us_leagues/csv/game_by_game/pure_luck/us_coin_flip_home_bias_v1.csv",
#     out_path="../../data/us_leagues/csv/standard_deviation/pure_luck/sd_win_rate_us_pure_luck_home_bias.csv",)

Wrote: /Users/joeyli/skillvsluck/output/european_soccer_leagues/points_std/actual.csv (rows: 4)
Wrote: /Users/joeyli/skillvsluck/output/european_soccer_leagues/points_std/skilled.csv (rows: 4)


In [ ]:
import pandas as pd

def league_std_winrate(in_path: str, out_path: str) -> pd.DataFrame:
    """
    Output (3 rows, one for each league): league, league_std_win_rate, rank

    Method:
      For each league, we compute the win rate for every team in every season.
      Then take thestandard deviationof those team season win rates to get the dispersion.
      Lastly leagues are ranked with highest being 1
    """
    df = pd.read_csv(in_path)
    need = {"season","date","league","home_team","away_team","result","score1","score2"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"Missing columns: {sorted(miss)}")

    # Per game to team perspective wins
    home = pd.DataFrame({"league": df["league"], "season": df["season"],
                         "team": df["home_team"], "win": (df["result"] == 1).astype(int)})
    away = pd.DataFrame({"league": df["league"], "season": df["season"],
                         "team": df["away_team"], "win": (df["result"] == -1).astype(int)})
    long = pd.concat([home, away], ignore_index=True)

    # Team season win rate
    team_season = (long
        .groupby(["league","season","team"], as_index=False)
        .agg(win_rate=("win","mean"))
    )

    # League dispersion 
    league_disp = (team_season
        .groupby("league", as_index=False)
        .agg(league_std_win_rate=("win_rate", lambda x: x.std(ddof=0)))
    )

    # Rank: higher std first; tie-break by league name
    league_disp = (league_disp
        .sort_values(["league_std_win_rate","league"], ascending=[False, True])
        .reset_index(drop=True)
    )
    league_disp["rank"] = league_disp.index + 1

    league_disp.to_csv(out_path, index=False)
    print(f"Wrote: {out_path} (rows: {len(league_disp)})")
    return league_disp


In [38]:
# Actual
act = league_std_winrate(
    in_path="../../data/us_leagues/csv/game_by_game/actual/us_combined_data.csv",
    out_path="../../output/us_leagues/dispersion/actual/league_sd_us_actual.csv",)

Wrote: ../../output/us_leagues/dispersion/actual/league_sd_us_actual.csv (rows: 3)


In [39]:
# Pure Skill
p_skill = league_std_winrate(
    in_path="../../data/us_leagues/csv/game_by_game/pure_skill/us_combined_pure_skill.csv",
    out_path="../../output/us_leagues/dispersion/pure_skill/league_sd_us_pure_skill.csv",)


Wrote: ../../output/us_leagues/dispersion/pure_skill/league_sd_us_pure_skill.csv (rows: 3)


In [40]:
# Pure Luck (home bias v1)
p_luck = league_std_winrate(
    in_path="../../data/us_leagues/csv/game_by_game/pure_luck/us_coin_flip_home_bias_v1.csv",
    out_path="../../output/us_leagues/dispersion/pure_luck/league_sd_us_pure_luck_home_bias.csv",)

Wrote: ../../output/us_leagues/dispersion/pure_luck/league_sd_us_pure_luck_home_bias.csv (rows: 3)
